# Use Case 1 - Revenue & Growth Analytics

> Exploration, validation and business-question answers for **Use Case 1 - Revenue & Growth Analytics (Finance pillar)**. Runs against the Bronze, Silver, and Gold tables produced by `transformations/silver/revenue/revenue_*.py` and `transformations/gold/revenue_kpis.py`.

## Business questions answered below
1. Monthly revenue trend - total, MoM %, YoY %, rolling 3m
2. Revenue by product category
3. Revenue by customer state
4. Payment-type mix & AOV
5. Executive summary (single-row KPI snapshot)

## 1. Sanity checks on Bronze

Confirm the four tables UC1 depends on are present and populated.

In [0]:
%sql
-- EXPECTED: every count > 0 (bronze ingestion populated all five source tables).
SELECT 'raw_orders'          AS tbl, COUNT(*) AS rows FROM data_sentinals.bronze.raw_orders          UNION ALL
SELECT 'raw_order_items'     , COUNT(*)               FROM data_sentinals.bronze.raw_order_items     UNION ALL
SELECT 'raw_order_payments'  , COUNT(*)               FROM data_sentinals.bronze.raw_order_payments  UNION ALL
SELECT 'raw_products'        , COUNT(*)               FROM data_sentinals.bronze.raw_products        UNION ALL
SELECT 'raw_product_names_translation', COUNT(*)      FROM data_sentinals.bronze.raw_product_names_translation

### 1a. Negative-payment check (spec requirement)

Count how many rows we expect the silver layer to drop. Running this **before** silver is built shows the raw quality of the source.

In [0]:
%sql
-- EXPECTED: negative_payments > 0 in raw Bronze (proves we need the silver DQ rule).
-- These rows get dropped by @dlt.expect_or_drop in silver/revenue/revenue_payments.py.
SELECT
  SUM(CASE WHEN payment_value < 0 THEN 1 ELSE 0 END) AS negative_payments,
  SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS null_payments,
  MIN(payment_value) AS min_value,
  MAX(payment_value) AS max_value
FROM data_sentinals.bronze.raw_order_payments

### 1b. Ghost-order check on payments

Payments whose `order_id` does not exist in `raw_orders` are "ghost" payments; inner join in silver should eliminate them.

In [0]:
%sql
-- EXPECTED: ghost_payment_rows may be > 0 in Bronze (these are payments whose
-- order_id has no matching line item). The INNER JOIN in silver/revenue/revenue_orders.py
-- eliminates them before Gold - see assertion 2 in the smoke-test cell at the end.
SELECT COUNT(*) AS ghost_payment_rows
FROM data_sentinals.bronze.raw_order_payments p
LEFT ANTI JOIN data_sentinals.bronze.raw_orders o
  ON p.order_id = o.order_id

## 2. Silver validations

After the DLT pipeline has run the silver tables, run these spot checks.

In [0]:
%sql
-- 2.1  No negative payments survived into silver
-- EXPECTED: 0.  Proof the silver DQ rule actually fires.
SELECT COUNT(*) AS negative_payments_in_silver
FROM data_sentinals.silver.payments_clean_silver
WHERE payment_value < 0

In [0]:
%sql
-- 2.2  Order-grain uniqueness for the payments rollup
-- EXPECTED: rows == unique_orders (exactly one row per order_id after the
-- 1-to-many collapse). Any difference means the aggregation broke.
SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS unique_orders
FROM data_sentinals.silver.payments_order_agg_silver

In [0]:
%sql
-- 2.3  Payment-mix distribution - sanity check on primary_payment_type
SELECT primary_payment_type, COUNT(*) AS orders, ROUND(SUM(total_payment_value), 2) AS total_value
FROM data_sentinals.silver.payments_order_agg_silver
GROUP BY primary_payment_type
ORDER BY orders DESC

In [0]:
%sql
-- 2.4  Revenue fact - row count, coverage, reconciliation
-- EXPECTED: recognised_orders ~= 0.97 * orders (the rest are canceled/unavailable),
--           pct_reconciled_within_5pct >= 95 (most orders have payments tying out
--           to the item revenue within 5%; vouchers/partials explain the tail).
SELECT
  COUNT(*)                                                AS orders,
  SUM(is_revenue_recognised)                              AS recognised_orders,
  ROUND(SUM(revenue_total), 2)                            AS revenue_total,
  ROUND(SUM(total_payment_value), 2)                      AS payments_total,
  ROUND(AVG(ABS(payment_reconciliation_delta)), 2)        AS avg_abs_recon_delta,
  ROUND(AVG(CASE WHEN ABS(payment_reconciliation_delta) / GREATEST(revenue_total, 1) < 0.05 THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_reconciled_within_5pct
FROM data_sentinals.silver.orders_revenue_silver

In [0]:
%sql
-- 2.5  Category coverage - should be very little 'unknown'
-- EXPECTED: unknown_pct < 5.  Evidence that the Portuguese->English translation
-- coalesce in silver/revenue/revenue_product_catalog.py covers the vast majority of products.
SELECT
  SUM(CASE WHEN product_category_en = 'unknown' THEN 1 ELSE 0 END) AS unknown_items,
  COUNT(*) AS total_items,
  ROUND(SUM(CASE WHEN product_category_en = 'unknown' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS unknown_pct
FROM data_sentinals.silver.order_items_revenue_silver

## 3. Business Question 1 - Monthly revenue trend (MoM, YoY, rolling 3m)

This is the hero chart of the use case. The Gold mart has all the window calculations pre-computed.

In [0]:
%sql
SELECT
  order_purchase_month,
  total_orders,
  ROUND(total_revenue, 2)          AS total_revenue,
  ROUND(aov, 2)                    AS aov,
  ROUND(revenue_mom_pct, 2)        AS mom_pct,
  ROUND(revenue_yoy_pct, 2)        AS yoy_pct,
  ROUND(revenue_rolling_3m, 2)     AS rolling_3m,
  ROUND(revenue_rolling_6m, 2)     AS rolling_6m
FROM data_sentinals.gold.gold_revenue_monthly
ORDER BY order_purchase_month

## 4. Business Question 2 - Revenue by product category

Which categories drive the business, and is the mix shifting over time?

In [0]:
%sql
-- Top 10 categories all-time
SELECT product_category_en,
       ROUND(SUM(total_revenue), 2) AS total_revenue,
       SUM(total_orders)            AS total_orders
FROM data_sentinals.gold.gold_revenue_by_category_month
GROUP BY product_category_en
ORDER BY total_revenue DESC
LIMIT 10

## 5. Business Question 3 - Revenue by customer state

Where is revenue concentrated? Useful for regional marketing and logistics planning.

In [0]:
%sql
SELECT customer_state,
       ROUND(SUM(total_revenue), 2) AS total_revenue,
       SUM(total_orders)            AS total_orders,
       ROUND(AVG(aov), 2)           AS avg_aov
FROM data_sentinals.gold.gold_revenue_by_state_month
GROUP BY customer_state
ORDER BY total_revenue DESC

## 6. Business Question 4 - Payment-type mix & AOV

Validates the 1-to-many payments rollup done in silver and surfaces AOV per payment method.

In [0]:
%sql
SELECT primary_payment_type,
       ROUND(SUM(total_revenue), 2)          AS total_revenue,
       SUM(total_orders)                     AS total_orders,
       ROUND(AVG(avg_installments), 2)       AS avg_installments,
       ROUND(AVG(share_of_month_revenue) * 100, 2) AS avg_share_of_month_pct
FROM data_sentinals.gold.gold_revenue_by_payment_type_month
GROUP BY primary_payment_type
ORDER BY total_revenue DESC

## 7. Business Question 5 - Executive summary

Single-row KPI snapshot covering total revenue, orders, AOV, peak month, and top category / state / payment method.

In [0]:
%sql
SELECT * FROM data_sentinals.gold.gold_revenue_exec_summary

## 8. Visual summary

Four inline charts over the Gold marts covering the headline monthly trend, top categories, top customer states, and payment-type mix.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

CATALOG = "data_sentinals"
plt.rcParams.update({
    "figure.figsize": (11, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PRIMARY, ACCENT = "#2E5EAA", "#E07A5F"

def _fmt_money(v, _pos=None):
    if v >= 1e6:
        return f"R$ {v/1e6:.1f}M"
    if v >= 1e3:
        return f"R$ {v/1e3:.0f}K"
    return f"R$ {v:.0f}"

money_fmt = FuncFormatter(_fmt_money)

monthly = spark.sql(f"""
    SELECT order_purchase_month, total_revenue, revenue_mom_pct
    FROM {CATALOG}.gold.gold_revenue_monthly
    ORDER BY order_purchase_month
""").toPandas()
monthly["month_label"] = monthly["order_purchase_month"].astype(str).str[:7]

fig, ax1 = plt.subplots()
ax1.bar(monthly["month_label"], monthly["total_revenue"],
        color=PRIMARY, width=0.7, label="Revenue")
ax1.set_ylabel("Revenue", color=PRIMARY)
ax1.set_xlabel("Month")
ax1.yaxis.set_major_formatter(money_fmt)
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly["month_label"], monthly["revenue_mom_pct"],
         color=ACCENT, marker="o", linewidth=2, label="MoM %")
ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax2.set_ylabel("MoM %", color=ACCENT)
ax2.grid(False)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left", frameon=False)

plt.title("Monthly Revenue and MoM Growth", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
top_cat = spark.sql(f"""
    SELECT product_category_en, ROUND(SUM(total_revenue), 0) AS revenue
    FROM {CATALOG}.gold.gold_revenue_by_category_month
    GROUP BY product_category_en
    ORDER BY revenue DESC
    LIMIT 10
""").toPandas().iloc[::-1]

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.barh(top_cat["product_category_en"], top_cat["revenue"], color=PRIMARY)
ax.xaxis.set_major_formatter(money_fmt)
ax.set_xlabel("Revenue")
ax.set_title("Top 10 Product Categories by Revenue", fontsize=13, pad=12)
ax.set_xlim(right=top_cat["revenue"].max() * 1.15)

for bar, val in zip(bars, top_cat["revenue"]):
    ax.text(val, bar.get_y() + bar.get_height() / 2,
            f"  {_fmt_money(val)}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
states = spark.sql(f"""
    SELECT customer_state, ROUND(SUM(total_revenue), 0) AS revenue
    FROM {CATALOG}.gold.gold_revenue_by_state_month
    GROUP BY customer_state
    ORDER BY revenue DESC
    LIMIT 10
""").toPandas()

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(states["customer_state"], states["revenue"], color=PRIMARY)
ax.yaxis.set_major_formatter(money_fmt)
ax.set_ylabel("Revenue")
ax.set_title("Top 10 Customer States by Revenue", fontsize=13, pad=12)
ax.set_ylim(top=states["revenue"].max() * 1.15)

for bar, val in zip(bars, states["revenue"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val,
            _fmt_money(val), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
pay = spark.sql(f"""
    SELECT primary_payment_type, ROUND(SUM(total_revenue), 0) AS revenue
    FROM {CATALOG}.gold.gold_revenue_by_payment_type_month
    GROUP BY primary_payment_type
    ORDER BY revenue DESC
""").toPandas()

palette = ["#2E5EAA", "#E07A5F", "#3D9970", "#8E44AD", "#F4A261"]
plt.figure(figsize=(6.5, 5))
plt.pie(pay["revenue"], labels=pay["primary_payment_type"], autopct="%1.1f%%",
        colors=palette[:len(pay)],
        wedgeprops={"width": 0.4, "edgecolor": "white"}, pctdistance=0.82)
plt.title("Revenue by Payment Type", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 9. UC1 smoke test (runnable assertions)

End-to-end data-quality gate for Use Case 1. Executes a single cell of SQL
invariants over the Silver and Gold layers and prints a `PASS` / `FAIL` per check.

Use it as the post-pipeline verification step: run it after every DLT refresh
and before any downstream consumer (dashboard, scheduled report, export) reads
the Gold marts. A red line means Gold is not safe to publish - the failing
assertion name points to the exact invariant that broke.

The checks codify the three UC1 spec requirements - no negative payments,
no ghost orders, 1-to-many payments collapse cleanly to order grain - plus
four shape contracts on the Gold marts: share columns sum to 1 per month,
exec summary is a single row, and category coverage stays above 95%.

In [ ]:
CATALOG = "data_sentinals"

invariants = [
    (
        "silver_no_negative_payments",
        f"SELECT COUNT(*) FROM {CATALOG}.silver.payments_clean_silver WHERE payment_value < 0",
        0,
        "==",
    ),
    (
        "silver_payments_rollup_is_order_grain",
        f"""SELECT COUNT(*) - COUNT(DISTINCT order_id)
              FROM {CATALOG}.silver.payments_order_agg_silver""",
        0,
        "==",
    ),
    (
        "silver_no_ghost_orders_in_fact",
        f"""SELECT COUNT(*)
              FROM {CATALOG}.silver.orders_revenue_silver f
              LEFT ANTI JOIN {CATALOG}.bronze.raw_orders o
                ON f.order_id = o.order_id""",
        0,
        "==",
    ),
    (
        "silver_category_unknown_rate_under_5pct",
        f"""SELECT ROUND(100.0 * SUM(CASE WHEN product_category_en = 'unknown' THEN 1 ELSE 0 END) / COUNT(*), 2)
              FROM {CATALOG}.silver.order_items_revenue_silver""",
        5.0,
        "<",
    ),
    (
        "gold_category_share_sums_to_one_per_month",
        f"""SELECT COUNT(*) FROM (
              SELECT order_purchase_month,
                     ROUND(SUM(revenue_share_of_month), 2) AS s
              FROM {CATALOG}.gold.gold_revenue_by_category_month
              GROUP BY 1
            ) WHERE s <> 1.00""",
        0,
        "==",
    ),
    (
        "gold_state_share_sums_to_one_per_month",
        f"""SELECT COUNT(*) FROM (
              SELECT order_purchase_month,
                     ROUND(SUM(revenue_share_of_month), 2) AS s
              FROM {CATALOG}.gold.gold_revenue_by_state_month
              GROUP BY 1
            ) WHERE s <> 1.00""",
        0,
        "==",
    ),
    (
        "gold_payment_type_share_sums_to_one_per_month",
        f"""SELECT COUNT(*) FROM (
              SELECT order_purchase_month,
                     ROUND(SUM(share_of_month_revenue), 2) AS s
              FROM {CATALOG}.gold.gold_revenue_by_payment_type_month
              GROUP BY 1
            ) WHERE s <> 1.00""",
        0,
        "==",
    ),
    (
        "gold_exec_summary_is_single_row",
        f"SELECT COUNT(*) FROM {CATALOG}.gold.gold_revenue_exec_summary",
        1,
        "==",
    ),
]

print(f"Running {len(invariants)} UC1 invariants against catalog '{CATALOG}'\n")
passed, failed = 0, 0
for name, query, expected, op in invariants:
    got = spark.sql(query).first()[0]
    ok = (got == expected) if op == "==" else (got < expected) if op == "<" else False
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}]  {name:<48s}  expected {op} {expected}, got {got}")
    passed += int(ok)
    failed += int(not ok)

print(f"\nSummary: {passed} passed, {failed} failed")
assert failed == 0, f"{failed} UC1 invariant(s) failed - Gold is not safe to publish"